# Base 1 (Raw Baseline Data) Training Orchestrator

This notebook performs sparse repository cloning, installs dependencies in editable mode, runs unit tests, and executes the Base 1 training pipeline on Dual Tesla T4 GPUs.

### Key Features:
- **Checkpoint Resume**: Automatically downloads existing `last.pt` from Google Drive to resume interrupted Kaggle sessions.
- **Incremental Real-Time Sync**: Synchronizes `results.csv`, `best.pt`, and `last.pt` to Google Drive at the end of **every single epoch** via callback.
- **Fast Dev Run Mode**: Run `--fast-dev-run` to execute a 30-second smoke test (1 epoch on 1% dataset) before launching full training.

## 1. Clone Repository & Install Package Dependencies

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'
BRANCH_NAME = '12-training-base-1'

# --- Environment Detection: Kaggle vs Colab ---
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
%cd {BASE_DIR}

REPO_PATH = Path(BASE_DIR) / REPO_NAME

# 1. Sparse clone repository
if not REPO_PATH.exists():
    print(f"Cloning {REPO_NAME} (branch {BRANCH_NAME})...")
    !git clone -q --depth 1 --branch {BRANCH_NAME} --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"Updating existing {REPO_NAME} repository...")
    %cd {REPO_NAME}
    !git checkout {BRANCH_NAME}
    !git pull -q origin {BRANCH_NAME}
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Directory navigation failed. Current path: {current_dir}")

# 3. Install package in editable mode
print("Installing package in editable mode with [cloud] dependencies...")
%pip install -q -e .[cloud]


## 2. Run Unit Tests

Execute co-located unit tests (BaseTrainingPipeline and Base1Trainer) to ensure system stability before launching training.

In [ ]:
# Run unit tests to verify training pipeline integrity
!pytest src/training/ -v


## 3. Execute Base 1 Training Pipeline

### Option A: Fast Dev Run (Smoke Test — ~30 seconds)
Runs 1 single epoch on 1% of the dataset to verify GPU acceleration, data loading, and Google Drive sync.

### Option B: Full Production Training
Runs full training schedule (100 epochs, Dual Tesla T4 GPUs, Early Stopping patience=20).

In [ ]:
# Option A: Fast Smoke Test (Uncomment to run a 30-second dry run)
# !python -m src.training.trainers.train_base_1 --fast-dev-run


In [ ]:
# Option B: Full Production Training (100 epochs, automatic Drive checkpoint resume)
!python -m src.training.trainers.train_base_1
